# 먹을 결심 v4 — 원리와 실제 출처
영양 API → 재료별 영양값 / 실제 레시피 사실 → RAG / Python → 끼니별 분량·재고 / LangGraph → 흐름 / Gradio → 화면.
인터넷 문서를 임의로 작성하지 않습니다. 검증한 원문 사실과 앱의 분량 조정을 구분합니다.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'pyproject.toml').exists(): raise RuntimeError('프로젝트 폴더에서 노트북을 열어주세요.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from fridge_protein.config import load_settings,TITLE
print(TITLE)
print(load_settings())  # 비밀키 값은 출력하지 않습니다.


## 1. 실제 출처 발췌 확인
검증일과 원문 URL, 원문 재료를 보세요. 이들은 가짜 테스트용 레시피가 아닙니다.
전체 조리법·사진은 배포하지 않으며, 짧은 발췌와 재료 사실만 포함합니다.
원문의 꼬집·큰술을 정확한 g으로 안다고 가정하지 않고 앱 조정량을 따로 표시합니다.

In [ ]:
from fridge_protein.verified_recipes import load_verified_documents
docs,warnings=load_verified_documents()
for d in docs:
    print(d.metadata['label'],d.metadata['url'],d.metadata['load_status'])
    print('원문 재료:',d.metadata['ingredients'])
    print('분량 조정:',d.metadata['adjustments'])


## 2. 식품 데이터 선택
같은 이름이 들어가도 쿠키·찌개는 두부 원재료와 다릅니다. 일반 식재료 조건으로 API를 요청하고
이름·조리 상태를 검사해 최대 5개만 보여 줍니다. API 실패를 실제 영양값으로 표시하지 않습니다.

In [ ]:
from fridge_protein.food_picker import ingredient_score,pick_options
assert ingredient_score('두부','두부쿠키') is None
assert ingredient_score('두부','된장찌개_두부') is None
print([f.name for f in pick_options('두부','sample')[0]])


## 3. 전체 그래프 실행
체중 → 실제 출처 검색 → 재료 확인 → 한 끼 최대 두 요리 조합 → 분량 조정 → 재고 검증 → 설명.
모델이 영양 수치를 만들지 않습니다. 후보 비율과 탐색 폭이 제한된 휴리스틱입니다.

In [ ]:
from fridge_protein.simple_ui import generate,DEFAULT_SELECTED
from fridge_protein.food_picker import selection_rows
from fridge_protein.workflow import build_graph
from fridge_protein.meal_planner import validate_plan
values=[60,'근육 증가 목표',3,DEFAULT_SELECTED,selection_rows(DEFAULT_SELECTED),True,50,20,20,4,'',False,'sample']
result,session,detail=generate(*values,None)
validate_plan(result['plan'])
print(build_graph().get_graph().draw_mermaid())
print(result['plan']['protein_g'],result['plan']['deficit_g'])


## 4. 입력 변경과 오래된 결과 방지
완료 결과의 입력 지문과 현재 입력을 비교합니다. 느린 API 요청이 뒤늦게 끝나도 이전 메뉴를 다시 표시하지 않습니다.

In [ ]:
from fridge_protein.simple_ui import render_current
changed=list(values);changed[2]=2
summary,cards,left,refs,raw=render_current(result,*changed)
assert cards=='' and raw=={}
print(summary)


## 5. 실제 외부 연결 확인 (선택)
실행 환경의 인터넷·공공 API 활용승인이 필요합니다. 이 셀을 실행한다고 유료 모델을 호출하지 않습니다.

In [ ]:
CHECK_CONNECTIONS=False
if CHECK_CONNECTIONS:
    from fridge_protein.nutrition import diagnose_food_api
    print(diagnose_food_api())
    docs,warnings=load_verified_documents(refresh=True)
    print('원문 준비',len(docs),'개',warnings)
else:print('외부 재조회 생략. 검증된 원문 발췌로 계산했습니다.')


## 테스트
`uv run pytest -q`. 정확한 외부/서버 검증 범위는 VALIDATION.md를 확인하세요.